#### ⏰ Deadline: Wednesday, July 1, 20:00
### What is this notebook about?

Many machine learning models produce probabilities, but can those probabilities be trusted?
Meet the concept of calibration! A model is well-calibrated if its predicted probabilities match observed outcomes. For example, among all patients assigned an 80% risk, we would expect about 80% to actually have the disease.
In this notebook you will get intuitions on what calibration is and how to achieve it. 

🚀 Time to get started! 

---
### **Important** 

Please note that the notebook contains images that will show when viewing them online in the Jupyter Book but may not show once you download the notebook. So, please consult the web version to view the images for stations that rely on them: https://mbp-lab.github.io/sose26-real-world-problems-ai/calibration/

---
#### 🟢 Station 1 - Why calibration matters

Let's start by considering a hypothetical scenario.

A hospital in a remote rural area has started using a machine learning model to estimate a patient's risk of having an undiagnosed disease. While the hospital benefits from modern AI tools, it still faces limited resources. The state only funds a limited number of diagnostic tests each month, so not every patient can be referred for additional screening.

As a result, only patients above a certain predicted risk threshold are referred. The historical dataset of the hospital is available to you.

- Load the dataset `rural_screening.csv` (fill in this step in the code below)
- It contains patient features, the true `disease` label and model `predicted_risk` scores from an undisclosed model
- We will apply a threshold of 0.5 and calculate model accuracy (already implemented below)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, confusion_matrix

# Load the dataset
# df = pd.read_csv("../data/rural_screening.csv") 

THRESHOLD = 0.5
y_true = df["disease"]
y_pred = (df["predicted_risk"] >= THRESHOLD).astype(int)

accuracy = accuracy_score(y_true, y_pred)
n_referred = y_pred.sum()

print(f"Patients: {len(df)}")
print(f"Disease prevalence: {y_true.mean():.1%}")
print(f"Threshold: {THRESHOLD}")
print(f"Patients referred: {n_referred} / {len(df)} ({100 * n_referred / len(df):.1f}%)")
print(f"Model accuracy: {accuracy:.3f}")

---
#### 🟢 Station 1 - Why calibration matters - continued

It seems that out of 3,000 patients, only a small fraction are referred for additional screening. Given the hospital's limited resources, this initially appears to be a sensible strategy.

However, the screening program has become increasingly concerned that the model may not be performing as well in practice. To investigate these concerns, the hospital hires a machine learning team to evaluate whether the model's predicted risks can actually be trusted.

The team leader requests visualizations (sketches below) to get a better picture of what's going on. 

- Your job is to implement the visualizations based on the code snippet below (fill in your part at the end of the code cell)
- But don't rush to the code yet! Look at the plots and write down what information you would need to create them.
- Find that information in the code snippet below. Some of the information is missing and you should fill it in!

![Caption](../images/sketch.jpg)




In [ ]:
# Some code to get started
# It has some of the information needed for the plots but not all!
# But you should be able to implement the missing parts easily based on what's already there

N_BINS = 7

# Divide the predicted risks into equally sized bins
df["risk_bin"] = pd.cut(
    df["predicted_risk"],
    bins=np.linspace(0, 1, N_BINS + 1),
    include_lowest=True)

grouped = df.groupby("risk_bin", observed=True)

# For each bin, compute the mean predicted risk, the observed disease rate and number of patients in each bin
calibration_table = pd.DataFrame({
    "mean_predicted_risk": grouped["predicted_risk"].mean(),
    #something's missing here,
    "n_patients": grouped["disease"].size()}).reset_index()


# Extract the columns as NumPy arrays for plotting
bin_centers = calibration_table["mean_predicted_risk"].to_numpy()
mean_predicted = calibration_table["mean_predicted_risk"].to_numpy()
#something's missing here
n_patients = calibration_table["n_patients"].to_numpy()

#something's missing here 

# -----------------------------
# TODO 
# Implement the suggested plots!

# -----------------------------

### 💡 Station 1 - Reflection

**Reminder**: A model is well-calibrated if its predicted probabilities match observed outcomes. For example, among all patients assigned an 80% risk, we would expect about 80% to actually have the disease.

- Why were these plots suggested? Which parts of the plots give information about the calibration quality?
- What do you observe? Is the model well-calibrated: does the predicted risk match the actual risk?
- What are the real-life consequences for a model like this? 👀Hint: think of the decision threshold!

---
#### 🟢🟢 Station 2 - Improving calibration

**ML team's decision on how to move on**

Based on the previous results, the ML team decides to recalibrate the model. Since they only have access to the model's predicted risk scores, not the underlying model itself, they use post-hoc calibration methods.

Post-hoc calibration means learning a correction to bring the original predicted risk closer to the actual risk! We cannot retrain the original model as it's not disclosed in our scenario. However, we can do something about those prediction scores. 

--- 
**Considered methods**

The post-hoc models they consider are:

**Temperature Scaling**: intuitively, it turns the model's confidence up or down by a fixed amount for all predictions. This is done by adjusting the predicted risks by first expressing them on the log-odds (logit) scale, where the logit is defined as $\text{logit}(p) = \log\left(\frac{p}{1-p}\right)$. These log-odds are then divided by a single parameter called the temperature $T$ and transformed back into probabilities. A temperature above 1 makes predictions less extreme, while a temperature below 1 makes them more extreme. The temperature is learned from the calibration set by finding the value that best matches predicted risks to observed disease rates.

**Isotonic Regression**: intuitively, it applies a second machine learning model to transform the original risk scores into better-calibrated probabilities. It fits a regression model that learns how predicted risks relate to observed disease rates. This relationship is constrained to be isotonic, meaning that regression preserves the ordering of the predictions. In Python, this can be implemented with `sklearn.isotonic.IsotonicRegression`. Note that we are training a new model, so we'll need a training or calibration sample and a validation (some call it test) sample!


- Check the code below and fill in the missing information to perform re-calibration.
- That will guide you to 1) run re-calibration 2) estimate how predicted probabilities match the observed ones (`brier_score_loss`) 3) visualize the results of the recalibration by re-using the sketches from Station 1!

In [ ]:
from scipy.optimize import minimize_scalar
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import brier_score_loss
from sklearn.model_selection import train_test_split

N_BINS = 7

# Split the data into a calibration set (80%) and a test set (20%)
cal_idx, test_idx = train_test_split(
    df.index,
    test_size=0.2,
    random_state=42,
    stratify=df["disease"],  # Keep the disease prevalence similar in both splits
)

raw_cal = df.loc[cal_idx, "predicted_risk"].to_numpy()
y_cal = df.loc[cal_idx, "disease"].to_numpy()

raw_test = df.loc[test_idx, "predicted_risk"].to_numpy()
y_test = df.loc[test_idx, "disease"].to_numpy()


# -----------------------------
# Temperature scaling
# -----------------------------
def logits_from_probabilities(probabilities, eps=1e-6):
    # Avoid probabilities of exactly 0 or 1, because log(p / (1 - p))
    # is undefined when p is 0 or 1.
    probabilities = np.clip(probabilities, eps, 1 - eps)
    return np.log(probabilities / (1 - probabilities))


def probabilities_from_logits(logits):
    # Convert logits back into probabilities between 0 and 1.
    return 1 / (1 + np.exp(-logits))


def apply_temperature(probabilities, temperature):
    # Divide the logits by the temperature.
    # temperature > 1 makes predictions less extreme.
    # temperature < 1 makes predictions more extreme.
    logits = logits_from_probabilities(probabilities)
    return probabilities_from_logits(logits / temperature)


def fit_temperature(probabilities, y):
    # Negative log-likelihood on the calibration set
    def loss(temperature):
        calibrated = apply_temperature(probabilities, temperature)

        # Avoid log(0), which is undefined.
        calibrated = np.clip(calibrated, 1e-6, 1 - 1e-6)

        return -np.mean(
            y * np.log(calibrated)
            + (1 - y) * np.log(1 - calibrated))

    # Directly optimize the single temperature parameter.
    # method="bounded" restricts the search to the interval below.
    # Here, the optimizer only tries temperatures between 0.05 and 20.
    result = minimize_scalar(loss,bounds=(0.05, 20),method="bounded" )

    return result.x


temperature = fit_temperature(raw_cal, y_cal)
temp_test = apply_temperature(raw_test, temperature)

print(f"Learned temperature: {temperature:.3f}")


# -----------------------------
# TODO 1
# Fit isotonic regression on the calibration set.

# As usual when training a model, we need to split the data into two parts
# a training or calibration set, used to learn the correction
# a validation set, used to evaluate whether the correction works on unseen patients.

# And obtain calibrated probabilities 

# -----------------------------



# -----------------------------
# TODO 2
# Compare the methods using the
# Brier score (already imported)
# -----------------------------
metrics = pd.DataFrame(
    {
        "Method": [
            "Original",
            "Temperature scaling",
            "Isotonic regression",
        ],
        "Brier score": [
            # brier_score_loss(...),
            # brier_score_loss(...),
            # brier_score_loss(...),], })

display(metrics.style.format(
        {"Brier score": "{:.4f}"}))


# -----------------------------
# TODO 3
# Recreate the calibration plots
# from Station 1 for:
#
# - Original predictions
# - Temperature scaling
# - Isotonic regression
# -----------------------------

### 💡 Station 2 - Reflection

- What happens after recalibration?
- What does the Brier score reflect?
   - 👀Look up how it's calculated: is it based on bins, data points? What values do you need to calculate it?

---

#### 🟢🟢🟢 Station 3 - Visualizing "confidence" 

The ML team now wants to communicate the model's predictions to the clinical team in a intuitive way. Rather than presenting technical metrics or tables, they decide to create simple visualizations. A useful way of doing this is visualizing the dataset as scatterplot (Feature 1 vs Feature 2). Next, one could create versions of this scatterplot by coloring each data point according to its predicted risk.

- Create the described visualizations for the original and recalibrated models.
- Write down *one* (and only one!) paragraph that would explain what the figures show to a non-technical audience! 

In [ ]:
# Your code here

### 💡 Station 3 - Reflection

Write down *one* (and only one!) paragraph that would explain what the figures show to a non-technical audience! 


---
####  Can the tutor add your visualizations to the class notes? A gallery of student results helps see how different people approach the problem.

[Optional] 🤔💭 Was there something you struggled with? Are there any open questions?